In [2]:
import pandas as pd

# Chargement des datasets
df = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\weatherdata.csv')
da = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\gender_submission.csv')
db = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\test.csv')
dc = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\train.csv')


In [ ]:
#EXERCICE 1

def supprimer_doublons(dataset, nom):
    print(f"\n{'='*50}")
    print(f"📊 {nom}")
    print(f"{'='*50}")

    lignes_avant = len(dataset)
    doublons = dataset.duplicated().sum()
    print(f"Lignes avant     : {lignes_avant}")
    print(f"Doublons trouvés : {doublons}")

    dataset_propre = dataset.drop_duplicates()

    lignes_apres = len(dataset_propre)
    print(f"Lignes après     : {lignes_apres}")
    print(f"Lignes supprimées: {lignes_avant - lignes_apres}")

    if doublons == 0:
        print("✅ Aucun doublon trouvé !")
    else:
        print(f"✅ {doublons} doublon(s) supprimé(s) !")

    return dataset_propre

df_propre = supprimer_doublons(df, "Weather Data")
da_propre = supprimer_doublons(da, "Gender Submission")
db_propre = supprimer_doublons(db, "Titanic Test")
dc_propre = supprimer_doublons(dc, "Titanic Train")

In [ ]:
#EXERCICE 2

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

dc = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\train.csv')

print("=== Valeurs manquantes ===")
print(dc.isnull().sum())
print(f"\nPourcentage manquant :")
print((dc.isnull().sum() / len(dc) * 100).round(2))

print("\n=== Stratégie 1 : Suppression ===")
lignes_avant = len(dc)

dc_suppression = dc.dropna(subset=['Embarked'])

lignes_apres = len(dc_suppression)
print(f"Lignes avant : {lignes_avant}")
print(f"Lignes après : {lignes_apres}")
print(f"Lignes supprimées : {lignes_avant - lignes_apres}")

print("\n=== Stratégie 2 : Imputation par la médiane (Age) ===")

imputer = SimpleImputer(strategy='median')
dc['Age'] = imputer.fit_transform(dc[['Age']])

print(f"Valeurs manquantes dans Age : {dc['Age'].isnull().sum()}")
print(f"Médiane utilisée : {dc['Age'].median()}")

print("\n=== Stratégie 3 : Remplissage par valeur constante (Embarked) ===")

mode_embarked = dc['Embarked'].mode()[0]
dc['Embarked'] = dc['Embarked'].fillna(mode_embarked)

print(f"Valeur utilisée : '{mode_embarked}'")
print(f"Valeurs manquantes dans Embarked : {dc['Embarked'].isnull().sum()}")

print("\n=== Stratégie 4 : Suppression de colonne (Cabin) ===")

print(f"Manquants dans Cabin : {dc['Cabin'].isnull().sum()} ({dc['Cabin'].isnull().mean()*100:.1f}%)")
dc = dc.drop(columns=['Cabin'])
print("✅ Colonne 'Cabin' supprimée (trop de valeurs manquantes)")

print("\n=== Vérification finale ===")
print(dc.isnull().sum())
print(f"\n✅ Dataset propre : {len(dc)} lignes × {len(dc.columns)} colonnes")


In [ ]:
#EXERCICE 3

import pandas as pd
from sklearn.preprocessing import LabelEncoder

dc = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\train.csv')

dc['FamilySize'] = dc['SibSp'] + dc['Parch'] + 1 

print("=== Family Size ===")
print(dc['FamilySize'].value_counts())

dc['IsAlone'] = (dc['FamilySize'] == 1).astype(int)

print("\n=== Is Alone ===")
print(dc['IsAlone'].value_counts())

dc['Title'] = dc['Name'].str.extract(r',\s*([^\.]+)\.')

print("\n=== Titres extraits ===")
print(dc['Title'].value_counts())

titres_rares = ['Lady', 'Countess', 'Capt', 'Col', 'Don',
                'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']

dc['Title'] = dc['Title'].replace(titres_rares, 'Rare')
dc['Title'] = dc['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

print("\n=== Titres simplifiés ===")
print(dc['Title'].value_counts())

print("\n=== Encodage One-Hot ===")

dc = pd.get_dummies(dc, columns=['Sex', 'Embarked'], drop_first=True)

print(dc[['Sex_male', 'Embarked_Q', 'Embarked_S']].head())

print("\n=== Encodage Label (Title) ===")

le = LabelEncoder()
dc['Title_encoded'] = le.fit_transform(dc['Title'])

print(dc[['Title', 'Title_encoded']].drop_duplicates().sort_values('Title_encoded'))

print("\n=== Nouvelles colonnes créées ===")
print(dc[['FamilySize', 'IsAlone', 'Title', 'Title_encoded',
          'Sex_male', 'Embarked_Q', 'Embarked_S']].head(10))

print(f"\n✅ Dataset final : {dc.shape[0]} lignes × {dc.shape[1]} colonnes")



In [ ]:
#EXERCICE 4

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

dc = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\train.csv')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dc['Age'].dropna().plot(kind='box', ax=axes[0,0], color='steelblue')
axes[0,0].set_title("Boxplot — Age (avant)")

dc['Fare'].plot(kind='box', ax=axes[0,1], color='orange')
axes[0,1].set_title("Boxplot — Fare (avant)")

dc['Age'].dropna().plot(kind='hist', ax=axes[1,0], bins=30, color='steelblue', edgecolor='white')
axes[1,0].set_title("Histogramme — Age (avant)")

dc['Fare'].plot(kind='hist', ax=axes[1,1], bins=30, color='orange', edgecolor='white')
axes[1,1].set_title("Histogramme — Fare (avant)")

plt.suptitle("Distributions AVANT traitement", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

def detecter_outliers_iqr(data, colonne):
    Q1  = data[colonne].quantile(0.25)
    Q3  = data[colonne].quantile(0.75)
    IQR = Q3 - Q1
    borne_inf = Q1 - 1.5 * IQR
    borne_sup = Q3 + 1.5 * IQR
    outliers  = data[(data[colonne] < borne_inf) | (data[colonne] > borne_sup)]

    print(f"\n=== IQR — {colonne} ===")
    print(f"Q1         : {Q1:.2f}")
    print(f"Q3         : {Q3:.2f}")
    print(f"IQR        : {IQR:.2f}")
    print(f"Borne inf  : {borne_inf:.2f}")
    print(f"Borne sup  : {borne_sup:.2f}")
    print(f"Outliers   : {len(outliers)}")
    return borne_inf, borne_sup

age_inf,  age_sup  = detecter_outliers_iqr(dc, 'Age')
fare_inf, fare_sup = detecter_outliers_iqr(dc, 'Fare')

def detecter_outliers_zscore(data, colonne, seuil=3):
    z_scores = np.abs(stats.zscore(data[colonne].dropna()))
    outliers = (z_scores > seuil).sum()
    print(f"\n=== Z-Score — {colonne} (seuil={seuil}) ===")
    print(f"Outliers détectés : {outliers}")

detecter_outliers_zscore(dc, 'Age')
detecter_outliers_zscore(dc, 'Fare')


print("\n=== Plafonnement Fare (quantile 0.98) ===")
print(f"Avant — Max Fare : {dc['Fare'].max():.2f}")

seuil_98 = dc['Fare'].quantile(0.98)
dc['Fare_capped'] = dc['Fare'].clip(upper=seuil_98)

print(f"Seuil 0.98       : {seuil_98:.2f}")
print(f"Après — Max Fare : {dc['Fare_capped'].max():.2f}")

print("\n=== Transformation Log — Fare ===")

dc['Fare_log'] = np.log1p(dc['Fare'])  

print(f"Fare original — Moyenne : {dc['Fare'].mean():.2f} | Std : {dc['Fare'].std():.2f}")
print(f"Fare log      — Moyenne : {dc['Fare_log'].mean():.2f} | Std : {dc['Fare_log'].std():.2f}")

print("\n=== Suppression outliers Age ===")
print(f"Lignes avant : {len(dc)}")

dc_clean = dc[(dc['Age'] >= age_inf) & (dc['Age'] <= age_sup)]

print(f"Lignes après : {len(dc_clean)}")
print(f"Supprimées   : {len(dc) - len(dc_clean)}")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

dc['Age'].dropna().plot(kind='box', ax=axes[0,0], color='steelblue')
axes[0,0].set_title("Age — Avant")

dc_clean['Age'].plot(kind='box', ax=axes[0,1], color='green')
axes[0,1].set_title("Age — Après suppression")

dc['Fare'].plot(kind='box', ax=axes[1,0], color='orange')
axes[1,0].set_title("Fare — Avant")

dc['Fare_capped'].plot(kind='box', ax=axes[1,1], color='green')
axes[1,1].set_title("Fare — Après plafonnement")

dc['Fare_log'].plot(kind='box', ax=axes[1,2], color='purple')
axes[1,2].set_title("Fare — Après log")

plt.suptitle("Comparaison AVANT / APRÈS traitement", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Comparaison statistiques Fare ===")
comparaison = pd.DataFrame({
    'Original' : dc['Fare'].describe(),
    'Plafonné' : dc['Fare_capped'].describe(),
    'Log'      : dc['Fare_log'].describe()
}).round(2)

print(comparaison)
print("\n✅ Traitement des valeurs aberrantes terminé !")


In [ ]:
#EXERCICE 5

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer

dc = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\train.csv')

dc['Age']  = SimpleImputer(strategy='median').fit_transform(dc[['Age']])
dc['Fare'] = dc['Fare'].clip(upper=dc['Fare'].quantile(0.98))
dc = dc.drop(columns=['Cabin', 'Name', 'Ticket'])

print("✅ Données prêtes pour la mise à l'échelle")
print(dc[['Age', 'Fare', 'SibSp', 'Parch']].describe().round(2))

features_std    = ['Age', 'SibSp', 'Parch']   
features_minmax = ['Fare']                     

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(['Age', 'Fare', 'SibSp', 'Parch']):
    dc[col].plot(kind='hist', bins=30, ax=axes[i],
                 color='steelblue', edgecolor='white')
    axes[i].set_title(f"{col} — Avant")

plt.suptitle("Distributions AVANT mise à l'échelle", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== StandardScaler ===")

scaler_std = StandardScaler()
dc[['Age_scaled', 'SibSp_scaled', 'Parch_scaled']] = scaler_std.fit_transform(
    dc[['Age', 'SibSp', 'Parch']]
)

print(dc[['Age_scaled', 'SibSp_scaled', 'Parch_scaled']].describe().round(2))

print("\n=== MinMaxScaler ===")

scaler_mm = MinMaxScaler()
dc[['Fare_scaled']] = scaler_mm.fit_transform(dc[['Fare']])

print(dc[['Fare_scaled']].describe().round(2))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, col in enumerate(['Age', 'Fare', 'SibSp', 'Parch']):
    dc[col].plot(kind='hist', bins=30, ax=axes[0, i],
                 color='steelblue', edgecolor='white')
    axes[0, i].set_title(f"{col} — Avant")

cols_apres = ['Age_scaled', 'Fare_scaled', 'SibSp_scaled', 'Parch_scaled']
couleurs   = ['green', 'orange', 'green', 'green']
methodes   = ['Standard', 'MinMax', 'Standard', 'Standard']

for i, (col, couleur, methode) in enumerate(zip(cols_apres, couleurs, methodes)):
    dc[col].plot(kind='hist', bins=30, ax=axes[1, i],
                 color=couleur, edgecolor='white')
    axes[1, i].set_title(f"{col.replace('_scaled','')} — {methode} (après)")

plt.suptitle("Comparaison AVANT / APRÈS mise à l'échelle", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== Résumé comparatif ===")

comparaison = pd.DataFrame({
    'Age_avant'      : dc['Age'].describe(),
    'Age_standard'   : dc['Age_scaled'].describe(),
    'Fare_avant'     : dc['Fare'].describe(),
    'Fare_minmax'    : dc['Fare_scaled'].describe(),
}).round(3)

print(comparaison)
print("\n✅ Mise à l'échelle terminée !")


In [ ]:
#EXERCICE 6

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

dc = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\train.csv')

dc['Age']  = SimpleImputer(strategy='median').fit_transform(dc[['Age']])
dc['Fare'] = dc['Fare'].clip(upper=dc['Fare'].quantile(0.98))
dc['Embarked'] = dc['Embarked'].fillna(dc['Embarked'].mode()[0])
dc = dc.drop(columns=['Cabin'])

dc['Title'] = dc['Name'].str.extract(r',\s*([^\.]+)\.')
titres_rares = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
dc['Title'] = dc['Title'].replace(titres_rares, 'Rare')
dc['Title'] = dc['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

print("✅ Pré-traitements appliqués")

print("\n=== Colonnes catégorielles restantes ===")
cat_cols = dc.select_dtypes(include=['object']).columns.tolist()
print(cat_cols)

print("\n=== Valeurs uniques par colonne ===")
for col in cat_cols:
    print(f"{col:<15} : {dc[col].nunique()} valeurs → {dc[col].unique()[:5]}")

print("\n=== One-Hot Encoding : Sex, Embarked ===")

dc = pd.get_dummies(dc, columns=['Sex', 'Embarked'], drop_first=True)

print("Colonnes créées :")
print([c for c in dc.columns if 'Sex' in c or 'Embarked' in c])
print(dc[['Sex_male', 'Embarked_Q', 'Embarked_S']].head())

print("\n=== Label Encoding : Title ===")

le = LabelEncoder()
dc['Title_encoded'] = le.fit_transform(dc['Title'])

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Mapping Title :")
for titre, code in mapping.items():
    print(f"  {titre:<10} → {code}")

dc = dc.drop(columns=['Name', 'Ticket', 'Title'])

print("\n=== Colonnes après encodage ===")
print(dc.columns.tolist())

print("\n=== Types de données finaux ===")
print(dc.dtypes)

print(f"\n=== Colonnes catégorielles restantes ===")
reste = dc.select_dtypes(include=['object']).columns.tolist()
print(reste if reste else "✅ Aucune — tout est numérique !")

print(f"\n✅ Dataset final : {dc.shape[0]} lignes × {dc.shape[1]} colonnes")
display(dc.head())

In [ ]:
#EXERCICE 7

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

dc = pd.read_csv(r'C:\Users\kmeso\Desktop\TTA_KOUAME_ASSE_SOKRY_JOSEPH\WEEK02\CVS\CVS2\train.csv')

dc['Age'] = SimpleImputer(strategy='median').fit_transform(dc[['Age']])

bins   = [0, 12, 18, 60, 100]
labels = ['Enfant', 'Adolescent', 'Adulte', 'Senior']

dc['AgeGroup'] = pd.cut(dc['Age'], bins=bins, labels=labels)

print("=== Groupes d'âge créés ===")
print(dc['AgeGroup'].value_counts())

print("\n=== Aperçu Age → AgeGroup ===")
print(dc[['Age', 'AgeGroup']].head(10))

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dc['Age'].plot(kind='hist', bins=30, ax=axes[0],
               color='steelblue', edgecolor='white')
axes[0].set_title("Distribution Age (brut)")
axes[0].set_xlabel("Age")

dc['AgeGroup'].value_counts().plot(kind='bar', ax=axes[1],
    color=['#4ADE80','#60A5FA','#F97316','#A78BFA'], edgecolor='white')
axes[1].set_title("Répartition par groupe d'âge")
axes[1].set_xlabel("Groupe")
axes[1].set_ylabel("Nombre de passagers")
axes[1].tick_params(rotation=0)

plt.suptitle("Transformation Age → Groupes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== One-Hot Encoding AgeGroup ===")

dc = pd.get_dummies(dc, columns=['AgeGroup'], prefix='Age')

cols_age = [c for c in dc.columns if c.startswith('Age_')]
print(f"Colonnes créées : {cols_age}")
print(dc[cols_age].head(10))

print("\n=== Taux de survie par groupe d'âge ===")

survie = pd.DataFrame({
    'Groupe'    : labels,
    'Total'     : [dc[f'Age_{l}'].sum() for l in labels],
    'Survivants': [dc[dc[f'Age_{l}'] == 1]['Survived'].sum() for l in labels],
})
survie['Taux survie (%)'] = (survie['Survivants'] / survie['Total'] * 100).round(1)
print(survie)

survie.plot(kind='bar', x='Groupe', y='Taux survie (%)',
            color=['#4ADE80','#60A5FA','#F97316','#A78BFA'],
            legend=False, figsize=(8, 5), edgecolor='white')
plt.title("Taux de survie par groupe d'âge")
plt.ylabel("Taux de survie (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n=== Dataset final ===")
print(dc[['Age'] + cols_age].head(10))
print(f"\n✅ Dataset : {dc.shape[0]} lignes × {dc.shape[1]} colonnes")
